# Part 2: Framework 1 – Ragas (Retrieval Augmented Generation Assessment)

Moving from theoretical metrics to implementation, Ragas is the leading open-source Python framework designed to turn "vibe checks" ("the answers look pretty good today") into hard, reproducible quantitative data.

Ragas maps each component of your pipeline to testable metrics using an evaluation dataset schema and runs them through LLM-as-a-judge scorers.

# 1. Core Architecture: The Evaluation Dataset Schema
To evaluate a RAG pipeline using Ragas, you construct a dataset containing four core columns (or lists):

**user_input (or question):** The prompt submitted by the user.

**retrieved_contexts:** A list of text chunks fetched by your retriever (Vector Store / Graph).

**response (or answer):** The final output generated by your LLM generator.

reference (optional, for ground truth): The human-approved ideal answer, used for calculating context recall and exact factual accuracy.

# 2. Implementation Code (ragas_evaluation_pipeline.py)
Here is a complete, clean Python implementation showing how to structure data and run an evaluation pipeline using Ragas:

In [ ]:
"""
ragas_evaluation_pipeline.py
Demonstrates setting up a test dataset schema and evaluating a RAG pipeline 
using Ragas core metrics (Faithfulness, Answer Relevancy, Context Recall).
"""

import os
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    LLMContextRecall,
    LLMContextPrecisionWithoutReference
)
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

def run_ragas_evaluation():
    # 1. Define your evaluation dataset (simulating RAG output records)
    evaluation_data = {
        "user_input": [
            "What caused the supply chain bottlenecks for TechCorp Europe in Q3?",
            "What role does DataStream Logistics play?"
        ],
        "retrieved_contexts": [
            [
                "TechCorp Europe experienced major supply chain bottlenecks in Q3 due to severe port strikes in Berlin and labor shortages.",
                "Global shipping rates fluctuated slightly during the quarter."
            ],
            [
                "DataStream Logistics provides tier-1 routing solutions and automated distribution networks across Germany."
            ]
        ],
        "response": [
            "TechCorp Europe's Q3 supply chain bottlenecks were caused by port strikes in Berlin and local labor shortages.",
            "DataStream Logistics handles tier-1 routing and automated distribution networks throughout Germany."
        ],
        "reference": [
            "Port strikes in Berlin and labor shortages caused the Q3 supply chain bottlenecks for TechCorp Europe.",
            "DataStream Logistics provides tier-1 routing solutions and automated distribution across Germany."
        ]
    }

    # Convert dictionary into a Hugging Face Dataset object (required by Ragas)
    dataset = Dataset.from_dict(evaluation_data)

    # 2. Set up the Evaluator LLM (GPT-4o acting as the judge)
    evaluator_model = ChatOpenAI(model="gpt-4o", temperature=0)
    wrapped_llm = LangchainLLMWrapper(evaluator_model)

    # 3. Instantiate Ragas Metrics and bind the evaluator model
    faithfulness_metric = Faithfulness(llm=wrapped_llm)
    relevancy_metric = ResponseRelevancy(llm=wrapped_llm)
    context_recall_metric = LLMContextRecall(llm=wrapped_llm)
    context_precision_metric = LLMContextPrecisionWithoutReference(llm=wrapped_llm)

    metrics_list = [
        faithfulness_metric,
        relevancy_metric,
        context_recall_metric,
        context_precision_metric
    ]

    print("Running Ragas Evaluation Suite...")

    # 4. Execute the evaluation
    results = evaluate(
        dataset=dataset,
        metrics=metrics_list
    )

    # 5. Output results dataframe
    df_results = results.to_pandas()
    print("\n--- Ragas Evaluation Scores ---")
    print(df_results[["user_input", "faithfulness", "response_relevancy", "llm_context_recall"]])
    
    # Print average aggregate scores
    print("\n--- Aggregate Performance ---")
    print(results)

if __name__ == "__main__":
    # Ensure OPENAI_API_KEY is set in your environment
    if "OPENAI_API_KEY" not in os.environ:
        print("Warning: Please ensure OPENAI_API_KEY is set in your environment variables.")
    else:
        run_ragas_evaluation()

## 3. Interpreting Ragas Scores for Debugging

When your evaluation script runs, each metric returns a normalized score between $0.0$ and $1.0$:

If faithfulness is low ($\le 0.7$) but context_recall is high: Your retriever fetched the correct information, but your prompt engineering or generator LLM is hallucinating unsupported details. Fix: Tighten your system prompt ("Answer strictly using only the provided context. Do not extrapolate").

If llm_context_recall is low ($\le 0.7$): Your retriever failed to find the necessary facts. Fix: Upgrade your chunking strategy, switch from pure vector search to hybrid search (RRF), or incorporate GraphRAG multi-hop traversal.